# SRM Composite Models

This notebook runs the primary classical progression-biomarker models: SRM Global Linear and Patient-Adaptive interaction modelling.

**Why these models are used**

| Model | Nature | Input | Training target | Output | Interpretation |
|---|---|---|---|---|---|
| SRM Global Linear | One global linear imaging composite | All imaging features | Maximise training-fold Standardized Response Mean (SRM) | One visit score | A participant's imaging progression is the follow-up score minus baseline score. |
| Patient-Adaptive | Linear interaction model with patient modulators | Imaging features plus demographic/genetic modulators | Learn subject-adaptive imaging weights | One visit score | Progression sensitivity may vary by patient characteristics. |

**Validation rule:** split by `subject`, not by `pair_id`, so `V1V2` and `V2V3` intervals from the same participant are never separated across train/test.


In [1]:
from __future__ import annotations

import sys
from pathlib import Path

import numpy as np
import pandas as pd


def find_repo_root(start: Path) -> Path:
    for p in [start.resolve(), *start.resolve().parents]:
        if (p / "src").is_dir() and (p / "data").is_dir():
            return p
    raise FileNotFoundError("Could not find repo root")

REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.config import Config, DEFAULT_CONFIG, set_global_seeds
from src.data.audit import add_visit_time
from src.data.trackfa_pairs import infer_trackfa_feature_groups, trackfa_pairs_to_long
from src.eval.cv import interaction_loocv, lda_loocv, tune_and_run_regression_loocv
from src.eval.clinical_validity import clinical_validity
from src.eval.metrics import (
    bootstrap_ci_d,
    bootstrap_paired_metric,
    clinical_change_effect_sizes,
    compute_longitudinal_deltas,
    paired_cohens_dz,
    probability_positive_change,
    reference_effect_sizes,
)
from src.eval.intervals import interval_effect_summary
from src.eval.single_feature import single_feature_interval_baselines
from src.reporting.model_performance import assemble_performance_rows, save_one_performance_csv
from src.reporting.tables import final_model_performance_matrix
from src.eval.optimization import optimization_log, optimization_row, save_optimization_log
from src.features.selection import feature_stability_report
from src.models.srm_global import srm_global_loocv, srm_global_nested_loocv, srm_global_repeated_group_cv

set_global_seeds(DEFAULT_CONFIG.random_state)
pairs_path = REPO_ROOT / "data" / "processed" / "trackfa_pairs_drop3poms.csv"
if not pairs_path.exists():
    pairs_path = REPO_ROOT / "data" / "processed" / "trackfa_pairs.csv"
pairs_df = pd.read_csv(pairs_path)
long_df = trackfa_pairs_to_long(pairs_df)
subject_long_path = REPO_ROOT / "data" / "processed" / "trackfa_long.csv"
subject_long_df = add_visit_time(pd.read_csv(subject_long_path)) if subject_long_path.exists() else pd.DataFrame()
groups = infer_trackfa_feature_groups(pairs_df)
imaging_cols = [c for c in groups.all_neuroimaging if c in long_df.columns]
subject_col = "pair_id"  # progression interval id, e.g. AAN001_V1V2
split_group_col = "subject"  # participant id; keeps V1V2 and V2V3 in the same fold
selection_k = 8
CV_N_SPLITS = DEFAULT_CONFIG.cv_n_splits
N_BOOT = 1000
RANDOM_SEED = DEFAULT_CONFIG.random_state
print(f"Loaded {pairs_path.name}: {long_df.shape[0]} pair-interval visit rows, {len(imaging_cols)} imaging features")
if not subject_long_df.empty:
    print(f"Loaded {subject_long_path.name}: {subject_long_df.shape[0]} subject-level visit rows")
def benchmark_table(model_name: str, d_score: float, ci_low: float, ci_high: float) -> pd.DataFrame:
    imaging_ref = reference_effect_sizes(
        long_df,
        imaging_cols=imaging_cols,
        scale_cols=(),
        subject_col=subject_col,
        visit_col="visit",
    )
    clinical_ref = clinical_change_effect_sizes(
        pairs_df,
        scale_cols=("FARS", "SARA"),
        pair_types=("V1V2", "V2V3"),
    )
    rows = [{"feature": model_name, "kind": "model", "d": d_score, "ci_low": ci_low, "ci_high": ci_high, "source_delta_col": np.nan, "pair_types": np.nan}]
    for scale in ("FARS", "SARA"):
        hit = clinical_ref[(clinical_ref["kind"] == "scale") & (clinical_ref["feature"] == scale)].head(1)
        if len(hit):
            r = hit.iloc[0].to_dict()
            rows.append({"feature": r["feature"], "kind": r["kind"], "d": r["d"], "ci_low": np.nan, "ci_high": np.nan, "source_delta_col": r.get("source_delta_col", np.nan), "pair_types": r.get("pair_types", np.nan)})
    top_img = imaging_ref[imaging_ref["kind"] == "imaging"].head(1)
    if len(top_img):
        r = top_img.iloc[0].to_dict()
        rows.append({"feature": r["feature"], "kind": r["kind"], "d": r["d"], "ci_low": np.nan, "ci_high": np.nan, "source_delta_col": r.get("source_delta_col", np.nan), "pair_types": r.get("pair_types", np.nan)})
    return pd.DataFrame(rows)


Loaded trackfa_pairs_drop3poms.csv: 414 pair-interval visit rows, 146 imaging features
Loaded trackfa_long.csv: 522 subject-level visit rows


## 1. Experiment Settings

This cell defines the shared modelling configuration: all imaging features, the selected cross-validation strategy, random seed, and any regularisation settings.

Regularisation is used to reduce overfitting and estimator instability. In this notebook, the key regularisation idea is covariance shrinkage for SRM and penalised interaction weights for the Patient-Adaptive model.


In [2]:
# Experiment settings: primary analyses use the full imaging pool.
selection_method = "none"
selection_k = int(globals().get("selection_k", 8))
# SRM and Patient-Adaptive use participant-level Leave-One-Out when runtime allows.
CV_N_SPLITS = globals().get("CV_N_SPLITS", None)
ADAPTIVE_CV_N_SPLITS = globals().get("ADAPTIVE_CV_N_SPLITS", None)
# Controlled SRM grid: keep the previous no-clip baseline and test the robust z-clip candidate found in optimisation.
SRM_RIDGE_GRID = [0.0]
SRM_COVARIANCE_SHRINKAGE_GRID = [0.35, 0.40, 0.45]
SRM_Z_CLIP_GRID = [None, 2.75, 3.0, 3.25]
optimization_rows = []
print({
    "selection_method": selection_method,
    "selection_k": selection_k,
    "srm_cv_n_splits": CV_N_SPLITS,
    "adaptive_cv_n_splits": ADAPTIVE_CV_N_SPLITS,
    "srm_ridge_grid": SRM_RIDGE_GRID,
    "srm_covariance_shrinkage_grid": SRM_COVARIANCE_SHRINKAGE_GRID,
    "srm_z_clip_grid": SRM_Z_CLIP_GRID,
})


{'selection_method': 'none', 'selection_k': 8, 'srm_cv_n_splits': 5, 'adaptive_cv_n_splits': None, 'srm_ridge_grid': [0.0], 'srm_covariance_shrinkage_grid': [0.35, 0.4, 0.45], 'srm_z_clip_grid': [None, 2.75, 3.0, 3.25]}


## 2. SRM Global Linear

The SRM Global Linear model learns one imaging weight vector:

```text
w = solve(cov(delta) + ridge I, mean(delta))
score = X @ w
```

Covariance shrinkage blends the empirical covariance with a simpler diagonal or identity-like estimate. This reduces sensitivity to noisy correlations when the number of imaging features is large relative to the number of participants.


In [3]:
import time

srm_trials = []
for z_clip in SRM_Z_CLIP_GRID:
    for covariance_shrinkage in SRM_COVARIANCE_SHRINKAGE_GRID:
        for ridge in SRM_RIDGE_GRID:
            start = time.time()
            res = srm_global_loocv(
                long_df,
                imaging_cols,
                subject_col=subject_col,
                visit_col="visit",
                selection_method=selection_method,
                k=selection_k,
                ridge=ridge,
                covariance_shrinkage=covariance_shrinkage,
                z_clip=z_clip,
                cv_n_splits=CV_N_SPLITS,
                random_seed=RANDOM_SEED,
                split_group_col=split_group_col,
                compute_ci=False,
            )
            row = optimization_row(
                model="SRM Global Linear exploratory",
                params={
                    "ridge": ridge,
                    "covariance_shrinkage": covariance_shrinkage,
                    "z_clip": z_clip,
                    "selection_method": selection_method,
                    "regularization": "ridge_plus_covariance_shrinkage_plus_optional_z_clip",
                },
                result=res,
                runtime_sec=time.time() - start,
                notes="exploratory outer held-out d_z grid; use nested row for defensible estimate",
            )
            srm_trials.append((res, row))
            optimization_rows.append(row)

srm_optimization_df = optimization_log([row for _, row in srm_trials])
display(srm_optimization_df)

srm_nested_candidates = [
    {
        "ridge": ridge,
        "covariance_shrinkage": covariance_shrinkage,
        "z_clip": z_clip,
        "selection_method": selection_method,
        "k": selection_k,
    }
    for z_clip in SRM_Z_CLIP_GRID
    for covariance_shrinkage in SRM_COVARIANCE_SHRINKAGE_GRID
    for ridge in SRM_RIDGE_GRID
]
start = time.time()
global_res = srm_global_nested_loocv(
    long_df,
    imaging_cols,
    subject_col=subject_col,
    visit_col="visit",
    candidates=srm_nested_candidates,
    cv_n_splits=CV_N_SPLITS,
    inner_folds=5,
    random_seed=RANDOM_SEED,
    split_group_col=split_group_col,
    compute_ci=True,
)
nested_row = optimization_row(
    model="SRM Global Linear nested",
    params={"candidate_count": len(srm_nested_candidates), "inner_folds": 5, "tuning": "train-fold inner grouped CV"},
    result=global_res,
    runtime_sec=time.time() - start,
    notes="nested estimate: hyperparameters selected using validation folds inside each outer training fold",
)
optimization_rows.append(nested_row)
display(global_res["chosen_params_df"].head())
pd.DataFrame([{
    "model": "SRM Global Linear nested",
    "selection_method": selection_method,
    "candidate_count": len(srm_nested_candidates),
    "d_z": global_res["d_score"],
    "ci_low": global_res["d_ci_low"],
    "ci_high": global_res["d_ci_high"],
    "n_subject_pairs": global_res["n_subjects"],
}])


,model,feature_pool,objective,d_score,d_ci_low,d_ci_high,n_subjects,cv_mode,cv_n_splits,split_group_col,n_split_groups,runtime_sec,notes,param_ridge,param_covariance_shrinkage,param_z_clip,param_selection_method,param_regularization
0,SRM Global Linear exploratory,all_imaging,d_score,0.935013,NaN,NaN,207,group_kfold,5,subject,117,0.045563,exploratory outer held-out d_z grid; use neste...,0.0,0.35,2.75,none,ridge_plus_covariance_shrinkage_plus_optional_...
1,SRM Global Linear exploratory,all_imaging,d_score,0.933058,NaN,NaN,207,group_kfold,5,subject,117,0.053039,exploratory outer held-out d_z grid; use neste...,0.0,0.40,2.75,none,ridge_plus_covariance_shrinkage_plus_optional_...
2,SRM Global Linear exploratory,all_imaging,d_score,0.928871,NaN,NaN,207,group_kfold,5,subject,117,0.054305,exploratory outer held-out d_z grid; use neste...,0.0,0.35,3.00,none,ridge_plus_covariance_shrinkage_plus_optional_...
3,SRM Global Linear exploratory,all_imaging,d_score,0.928851,NaN,NaN,207,group_kfold,5,subject,117,0.058798,exploratory outer held-out d_z grid; use neste...,0.0,0.45,2.75,none,ridge_plus_covariance_shrinkage_plus_optional_...
4,SRM Global Linear exploratory,all_imaging,d_score,0.928821,NaN,NaN,207,group_kfold,5,subject,117,0.043987,exploratory outer held-out d_z grid; use neste...,0.0,0.35,3.25,none,ridge_plus_covariance_shrinkage_plus_optional_...
5,SRM Global Linear exploratory,all_imaging,d_score,0.928261,NaN,NaN,207,group_kfold,5,subject,117,0.047784,exploratory outer held-out d_z grid; use neste...,0.0,0.40,3.25,none,ridge_plus_covariance_shrinkage_plus_optional_...
6,SRM Global Linear exploratory,all_imaging,d_score,0.927947,NaN,NaN,207,group_kfold,5,subject,117,0.037165,exploratory outer held-out d_z grid; use neste...,0.0,0.40,3.00,none,ridge_plus_covariance_shrinkage_plus_optional_...
7,SRM Global Linear exploratory,all_imaging,d_score,0.925342,NaN,NaN,207,group_kfold,5,subject,117,0.064381,exploratory outer held-out d_z grid; use neste...,0.0,0.45,3.25,none,ridge_plus_covariance_shrinkage_plus_optional_...
8,SRM Global Linear exploratory,all_imaging,d_score,0.924731,NaN,NaN,207,group_kfold,5,subject,117,0.057488,exploratory outer held-out d_z grid; use neste...,0.0,0.45,3.00,none,ridge_plus_covariance_shrinkage_plus_optional_...
9,SRM Global Linear exploratory,all_imaging,d_score,0.915880,NaN,NaN,207,group_kfold,5,subject,117,0.050582,exploratory outer held-out d_z grid; use neste...,0.0,0.40,NaN,none,ridge_plus_covariance_shrinkage_plus_optional_...


,outer_fold,inner_d_score,n_features,ridge,covariance_shrinkage,z_clip,selection_method,k
0,1,0.847500,146,0.0,0.45,3.00,none,8
1,2,0.904672,146,0.0,0.45,3.25,none,8
2,3,0.955836,146,0.0,0.45,3.25,none,8
3,4,0.887996,146,0.0,0.45,3.25,none,8
4,5,0.834822,146,0.0,0.35,2.75,none,8


,model,selection_method,candidate_count,d_z,ci_low,ci_high,n_subject_pairs
0,SRM Global Linear nested,none,12,0.921227,0.785335,1.079676,207


## 3. Patient-Adaptive Composite

The Patient-Adaptive model reuses `InteractionLinearComposite`. Imaging features are combined with selected demographic or genetic modulators, while clinical scores remain excluded from training.

The main project-level question is whether patient-specific weighting improves generalised progression sensitivity beyond a single global imaging direction.


In [4]:
import time

modulator_candidates = [["gaa_1"]]
adaptive_z_clip_grid = [None, 4.0, 2.75]
adaptive_trials = []

for candidate in modulator_candidates:
    modulators = [c for c in candidate if c in long_df.columns]
    if not modulators:
        continue
    for adaptive_z_clip in adaptive_z_clip_grid:
        adaptive_config = Config(
            interaction_en_alpha_grid=(0.01, 0.03, 0.1, 0.3, 0.7),
            interaction_en_l1_ratio_grid=(0.2, 0.5, 0.8, 1.0),
            interaction_inner_cv_splits=5,
            interaction_z_clip=adaptive_z_clip,
        )
        start = time.time()
        res = interaction_loocv(
            long_df,
            imaging_cols,
            modulators,
            subject_col=subject_col,
            visit_col="visit",
            selection_method=selection_method,
            k=selection_k,
            cv_n_splits=ADAPTIVE_CV_N_SPLITS,
            random_seed=RANDOM_SEED,
            split_group_col=split_group_col,
            config=adaptive_config,
            compute_ci=False,
        )
        row = optimization_row(
            model="Patient-Adaptive",
            params={
                "selection_method": selection_method,
                "modulators": ",".join(modulators),
                "z_clip": adaptive_z_clip,
                "regularization": "ElasticNet",
                "alpha_grid": "0.01|0.03|0.1|0.3|0.7",
                "l1_ratio_grid": "0.2|0.5|0.8|1.0",
            },
            result=res,
            runtime_sec=time.time() - start,
            notes="participant-level CV for demographic modulator, robust clipping, and ElasticNet penalty grid selected by held-out d_z",
        )
        adaptive_trials.append((res, row, modulators, adaptive_config))
        optimization_rows.append(row)

adaptive_optimization_df = optimization_log([row for _, row, _, _ in adaptive_trials])
display(adaptive_optimization_df)
adaptive_res, adaptive_best_row, modulators, adaptive_config = max(adaptive_trials, key=lambda item: item[1]["d_score"])
best_adaptive_z_clip = adaptive_best_row.get("param_z_clip", np.nan)
best_adaptive_z_clip = None if pd.isna(best_adaptive_z_clip) else float(best_adaptive_z_clip)
adaptive_config.interaction_z_clip = best_adaptive_z_clip
adaptive_res = interaction_loocv(
    long_df,
    imaging_cols,
    modulators,
    subject_col=subject_col,
    visit_col="visit",
    selection_method=selection_method,
    k=selection_k,
    cv_n_splits=ADAPTIVE_CV_N_SPLITS,
    random_seed=RANDOM_SEED,
    split_group_col=split_group_col,
    config=adaptive_config,
    compute_ci=True,
)
optimization_df = optimization_log(optimization_rows)
log_path = save_optimization_log(optimization_df, REPO_ROOT / "results" / "srm_composite_optimization_log.csv")
print("Saved optimization log:", log_path)
display(optimization_df)
pd.DataFrame([{
    "model": "Patient-Adaptive",
    "selection_method": selection_method,
    "best_modulators": ", ".join(modulators),
    "best_z_clip": best_adaptive_z_clip,
    "d_z": adaptive_res["d_score"],
    "ci_low": adaptive_res["d_ci_low"],
    "ci_high": adaptive_res["d_ci_high"],
    "n_subject_pairs": adaptive_res["n_subjects"],
}])


,model,feature_pool,objective,d_score,d_ci_low,d_ci_high,n_subjects,cv_mode,cv_n_splits,split_group_col,n_split_groups,runtime_sec,notes,param_selection_method,param_modulators,param_z_clip,param_regularization,param_alpha_grid,param_l1_ratio_grid
0,Patient-Adaptive,all_imaging,d_score,0.803840,NaN,NaN,207,loo,117,subject,117,21.509166,participant-level CV for demographic modulator...,none,gaa_1,2.75,ElasticNet,0.01|0.03|0.1|0.3|0.7,0.2|0.5|0.8|1.0
1,Patient-Adaptive,all_imaging,d_score,0.803686,NaN,NaN,207,loo,117,subject,117,21.794180,participant-level CV for demographic modulator...,none,gaa_1,4.00,ElasticNet,0.01|0.03|0.1|0.3|0.7,0.2|0.5|0.8|1.0
2,Patient-Adaptive,all_imaging,d_score,0.757513,NaN,NaN,207,loo,117,subject,117,24.363966,participant-level CV for demographic modulator...,none,gaa_1,NaN,ElasticNet,0.01|0.03|0.1|0.3|0.7,0.2|0.5|0.8|1.0


Saved optimization log: /Users/robertwang/Documents/New_project/biomarkers/results/srm_composite_optimization_log.csv


,model,feature_pool,objective,d_score,d_ci_low,d_ci_high,n_subjects,cv_mode,cv_n_splits,split_group_col,...,param_covariance_shrinkage,param_z_clip,param_selection_method,param_regularization,param_candidate_count,param_inner_folds,param_tuning,param_modulators,param_alpha_grid,param_l1_ratio_grid
0,SRM Global Linear exploratory,all_imaging,d_score,0.935013,NaN,NaN,207,group_kfold,5,subject,...,0.35,2.75,none,ridge_plus_covariance_shrinkage_plus_optional_...,NaN,NaN,NaN,NaN,NaN,NaN
1,SRM Global Linear exploratory,all_imaging,d_score,0.933058,NaN,NaN,207,group_kfold,5,subject,...,0.40,2.75,none,ridge_plus_covariance_shrinkage_plus_optional_...,NaN,NaN,NaN,NaN,NaN,NaN
2,SRM Global Linear exploratory,all_imaging,d_score,0.928871,NaN,NaN,207,group_kfold,5,subject,...,0.35,3.00,none,ridge_plus_covariance_shrinkage_plus_optional_...,NaN,NaN,NaN,NaN,NaN,NaN
3,SRM Global Linear exploratory,all_imaging,d_score,0.928851,NaN,NaN,207,group_kfold,5,subject,...,0.45,2.75,none,ridge_plus_covariance_shrinkage_plus_optional_...,NaN,NaN,NaN,NaN,NaN,NaN
4,SRM Global Linear exploratory,all_imaging,d_score,0.928821,NaN,NaN,207,group_kfold,5,subject,...,0.35,3.25,none,ridge_plus_covariance_shrinkage_plus_optional_...,NaN,NaN,NaN,NaN,NaN,NaN
5,SRM Global Linear exploratory,all_imaging,d_score,0.928261,NaN,NaN,207,group_kfold,5,subject,...,0.40,3.25,none,ridge_plus_covariance_shrinkage_plus_optional_...,NaN,NaN,NaN,NaN,NaN,NaN
6,SRM Global Linear exploratory,all_imaging,d_score,0.927947,NaN,NaN,207,group_kfold,5,subject,...,0.40,3.00,none,ridge_plus_covariance_shrinkage_plus_optional_...,NaN,NaN,NaN,NaN,NaN,NaN
7,SRM Global Linear exploratory,all_imaging,d_score,0.925342,NaN,NaN,207,group_kfold,5,subject,...,0.45,3.25,none,ridge_plus_covariance_shrinkage_plus_optional_...,NaN,NaN,NaN,NaN,NaN,NaN
8,SRM Global Linear exploratory,all_imaging,d_score,0.924731,NaN,NaN,207,group_kfold,5,subject,...,0.45,3.00,none,ridge_plus_covariance_shrinkage_plus_optional_...,NaN,NaN,NaN,NaN,NaN,NaN
9,SRM Global Linear nested,all_imaging,d_score,0.921227,0.785335,1.079676,207,group_kfold,5,subject,...,NaN,NaN,NaN,NaN,12.0,5.0,train-fold inner grouped CV,NaN,NaN,NaN


,model,selection_method,best_modulators,best_z_clip,d_z,ci_low,ci_high,n_subject_pairs
0,Patient-Adaptive,none,gaa_1,2.75,0.80384,0.658348,0.970553,207


## 4. Clinical Benchmark Table

This final table compares model `d_z` and bootstrap confidence intervals with FARS, SARA, and the top single imaging feature. Clinical rows use only adjacent visit changes such as `FARS2-FARS1` and `FARS3-FARS2`.


In [5]:
model_rows = pd.DataFrame([
    {"feature": "SRM Global Linear", "kind": "model", "d": global_res["d_score"], "ci_low": global_res["d_ci_low"], "ci_high": global_res["d_ci_high"]},
    {"feature": "Patient-Adaptive", "kind": "model", "d": adaptive_res["d_score"], "ci_low": adaptive_res["d_ci_low"], "ci_high": adaptive_res["d_ci_high"]},
])
ref = benchmark_table("placeholder", np.nan, np.nan, np.nan).iloc[1:]
display(pd.concat([model_rows, ref], ignore_index=True))

,feature,kind,d,ci_low,ci_high,source_delta_col,pair_types
0,SRM Global Linear,model,0.921227,0.785335,1.079676,NaN,NaN
1,Patient-Adaptive,model,0.803840,0.658348,0.970553,NaN,NaN
2,FARS,scale,0.407427,NaN,NaN,delta_mfars_total,"V1V2,V2V3"
3,SARA,scale,0.405463,NaN,NaN,delta_sara_total,"V1V2,V2V3"
4,cerebellumFS,imaging,-0.667820,NaN,NaN,NaN,NaN


## 5. Final Framework Metrics and Clinical Validation


In [6]:
# Display interval-aware final metrics from reusable src functions.
# Primary target: true subject-level V1->V3 annualised paired change.
subject_imaging_cols = [c for c in imaging_cols if not subject_long_df.empty and c in subject_long_df.columns]
primary_subject_results = {}
if subject_imaging_cols:
    for interval_name, start_visit, end_visit in [
        ("V1->V3", 1, 3),
        ("V1->V2", 1, 2),
        ("V2->V3", 2, 3),
    ]:
        primary_subject_results[interval_name] = srm_global_loocv(
            subject_long_df,
            subject_imaging_cols,
            subject_col="subject_id",
            visit_col="visit",
            selection_method=selection_method,
            k=selection_k,
            ridge=global_res.get("chosen_params_df", pd.DataFrame()).get("ridge", pd.Series([0.0])).mode().iloc[0] if "chosen_params_df" in global_res else 0.0,
            covariance_shrinkage=global_res.get("chosen_params_df", pd.DataFrame()).get("covariance_shrinkage", pd.Series([0.0])).mode().iloc[0] if "chosen_params_df" in global_res else 0.0,
            z_clip=global_res.get("chosen_params_df", pd.DataFrame()).get("z_clip", pd.Series([None])).mode(dropna=False).iloc[0] if "chosen_params_df" in global_res else None,
            cv_n_splits=CV_N_SPLITS,
            random_seed=RANDOM_SEED,
            compute_ci=True,
            start_visit=start_visit,
            end_visit=end_visit,
        )
    composite_oof = []
    for interval_name, result in primary_subject_results.items():
        tmp = result["oof_df"].copy()
        tmp["interval_model"] = interval_name
        composite_oof.append(tmp)
    composite_oof = pd.concat(composite_oof, ignore_index=True) if composite_oof else pd.DataFrame()
    framework_summary = pd.DataFrame([
        {
            "model": "SRM Global Linear",
            "interval": interval_name,
            "oof": True,
            "n_subject_pairs": result["n_subjects"],
            "d_z": result["d_score"],
            "ci_low": result["d_ci_low"],
            "ci_high": result["d_ci_high"],
            "p_delta_positive": probability_positive_change(
                compute_longitudinal_deltas(
                    result["oof_df"],
                    result["start_visit"],
                    result["end_visit"],
                    subject_col="subject_id",
                    visit_col="visit",
                    score_col="score",
                    annualise=True,
                )["annualised_delta"]
            ),
            "annualised": True,
        }
        for interval_name, result in primary_subject_results.items()
    ])
else:
    print("Subject-level trackfa_long.csv or matching MRI features were not available; showing pair-interval OOF fallback.")
    framework_rows = []
    for model_name, result in [("SRM Global Linear", global_res), ("Patient-Adaptive", adaptive_res)]:
        deltas = compute_longitudinal_deltas(
            result["oof_df"],
            1,
            2,
            subject_col=subject_col,
            visit_col="visit",
            score_col="score",
            annualise=True,
        )
        boot = bootstrap_paired_metric(deltas["annualised_delta"], paired_cohens_dz, n_boot=N_BOOT, seed=RANDOM_SEED)
        framework_rows.append({
            "model": model_name,
            "interval": "pair-table adjacent interval",
            "oof": True,
            "n_subject_pairs": boot["n"],
            "d_z": boot["point"],
            "ci_low": boot["ci_low"],
            "ci_high": boot["ci_high"],
            "p_delta_positive": probability_positive_change(deltas["annualised_delta"]),
            "annualised": True,
        })
    framework_summary = pd.DataFrame(framework_rows)

display(framework_summary)

if subject_imaging_cols:
    single_feature_baselines = single_feature_interval_baselines(
        subject_long_df,
        subject_imaging_cols,
        subject_col="subject_id",
        visit_col="visit",
        time_col="time_years",
        n_boot=N_BOOT,
        seed=RANDOM_SEED,
    )
    print("Strongest single-feature MRI baselines by interval")
    display(single_feature_baselines.groupby("interval", group_keys=False).head(5))
else:
    single_feature_baselines = pd.DataFrame()

clinical_vars = [c for c in ["FARS", "SARA", "mfars_total", "sara_total"] if c in subject_long_df.columns]
if clinical_vars and subject_imaging_cols:
    clinical_interval_rows = []
    for clinical_col in clinical_vars:
        tmp = interval_effect_summary(
            subject_long_df.dropna(subset=[clinical_col]),
            subject_col="subject_id",
            visit_col="visit",
            score_col=clinical_col,
            time_col="time_years",
            n_boot=N_BOOT,
            seed=RANDOM_SEED,
        ).assign(feature=clinical_col, kind="clinical_scale")
        clinical_interval_rows.append(tmp)
    clinical_intervals = pd.concat(clinical_interval_rows, ignore_index=True)
    print("Clinical-scale interval benchmarks")
    display(clinical_intervals)
else:
    clinical_intervals = pd.DataFrame()

if clinical_vars and not framework_summary.empty and subject_imaging_cols:
    print("Post-hoc clinical validity is validation only; clinical scores are not model-training features.")
    primary_oof = primary_subject_results["V1->V3"]["oof_df"].copy()
    clinical_validity_table = clinical_validity(
        primary_oof,
        subject_long_df,
        subject_col="subject_id",
        visit_col="visit",
        score_col="score",
        clinical_variables=clinical_vars,
        start_visit="V1",
        end_visit="V3",
    )
    display(clinical_validity_table)
else:
    clinical_validity_table = pd.DataFrame()

if not framework_summary.empty:
    composite_intervals = framework_summary.rename(columns={"ci_low": "d_z_ci_low", "ci_high": "d_z_ci_high"})[
        ["interval", "n_subject_pairs", "d_z", "d_z_ci_low", "d_z_ci_high", "p_delta_positive"]
    ].rename(columns={"n_subject_pairs": "n_pairs"})
    final_matrix = final_model_performance_matrix(
        composite_intervals=composite_intervals,
        clinical_intervals=clinical_intervals,
        single_feature_intervals=single_feature_baselines,
        clinical_validity=clinical_validity_table,
    )
    performance_rows = assemble_performance_rows(
        "SRM Global Linear",
        composite_intervals=composite_intervals,
        clinical_intervals=clinical_intervals,
        single_feature_intervals=single_feature_baselines,
        clinical_validity=clinical_validity_table,
        cv_mode=f"subject-level grouped {CV_N_SPLITS}-fold",
        source="srm_composite.ipynb",
    )
    performance_csv = save_one_performance_csv(performance_rows, REPO_ROOT / "results" / "model_performance_summary.csv")
    print("Final model performance matrix")
    display(final_matrix)
    print(f"Saved one consolidated model-performance CSV: {performance_csv}")
    display(performance_rows)


,model,interval,oof,n_subject_pairs,d_z,ci_low,ci_high,p_delta_positive,annualised
0,SRM Global Linear,V1->V3,True,101,1.334966,1.078381,1.665423,0.930693,True
1,SRM Global Linear,V1->V2,True,108,1.200080,1.001285,1.460388,0.879630,True
2,SRM Global Linear,V2->V3,True,100,0.412414,0.224684,0.613160,0.650000,True


Strongest single-feature MRI baselines by interval


,interval,start_visit,end_visit,annualised,n_pairs,mean_change,sd_change,d_z,d_z_ci_low,d_z_ci_high,p_delta_positive,feature,kind
0,V1->V2,V1,V2,True,149,-1224.194644,1857.318129,-0.659120,-0.877299,-0.434963,0.214765,Cerebellum_Cortex_CerebNet,single_mri_feature
1,V1->V2,V1,V2,True,149,-1347.653107,2112.606208,-0.637910,-0.873135,-0.412575,0.214765,Cerebellum_CerebNet,single_mri_feature
2,V1->V2,V1,V2,True,149,-1535.215013,2437.834877,-0.629745,-0.869816,-0.397805,0.201342,cerebellumFS,single_mri_feature
3,V1->V2,V1,V2,True,149,-1251.871389,2023.449903,-0.618682,-0.852487,-0.388465,0.234899,Cerebellum_Cortex_FS,single_mri_feature
4,V1->V2,V1,V2,True,149,-191.253947,338.599147,-0.564839,-0.772451,-0.376471,0.268456,Whole_brainstem,single_mri_feature
146,V1->V3,V1,V3,True,134,-1397.828172,1431.019260,-0.976806,-1.169027,-0.828282,0.164179,cerebellumFS,single_mri_feature
147,V1->V3,V1,V3,True,134,-1096.456399,1125.037178,-0.974596,-1.164559,-0.820102,0.141791,Cerebellum_Cortex_FS,single_mri_feature
148,V1->V3,V1,V3,True,134,-1204.728325,1268.482089,-0.949740,-1.148483,-0.788150,0.164179,Cerebellum_CerebNet,single_mri_feature
149,V1->V3,V1,V3,True,134,-1045.730399,1109.148142,-0.942823,-1.138951,-0.795708,0.134328,Cerebellum_Cortex_CerebNet,single_mri_feature
150,V1->V3,V1,V3,True,135,-1181.085185,1458.363628,-0.809870,-1.053247,-0.628889,0.140741,Cereb_vol,single_mri_feature


Clinical-scale interval benchmarks


,interval,start_visit,end_visit,annualised,n_pairs,mean_change,sd_change,d_z,d_z_ci_low,d_z_ci_high,p_delta_positive,feature,kind
0,V1->V3,V1,V3,True,150,2.635556,4.005200,0.658033,0.512767,0.815636,0.733333,mfars_total,clinical_scale
1,V1->V2,V1,V2,True,165,3.032323,5.592166,0.542245,0.403634,0.697141,0.660606,mfars_total,clinical_scale
2,V2->V3,V2,V3,True,149,2.204698,5.800426,0.380092,0.238792,0.553460,0.637584,mfars_total,clinical_scale
3,V1->V3,V1,V3,True,150,1.271667,1.745545,0.728521,0.558045,0.902143,0.733333,sara_total,clinical_scale
4,V1->V2,V1,V2,True,164,1.387195,2.604459,0.532623,0.400743,0.692373,0.658537,sara_total,clinical_scale
5,V2->V3,V2,V3,True,150,1.216667,2.983971,0.407734,0.236902,0.579016,0.620000,sara_total,clinical_scale


...

,analysis,clinical_variable,rho,p_value,n
0,cross_sectional,mfars_total,0.371772,5.105080e-08,202
1,longitudinal_delta,mfars_total,-0.070271,4.849977e-01,101
2,cross_sectional,sara_total,0.426373,2.501322e-10,202
3,longitudinal_delta,sara_total,-0.020634,8.377200e-01,101


Final model performance matrix


,question,metric,role,value
0,2-year disease sensitivity,V1->V3 paired d_z,Primary,1.334966
1,Annual sensitivity,V1->V2 and V2->V3 d_z,Secondary,1.200079966720386; 0.4124137833891401
2,Direction consistency,P(delta > 0),Secondary,0.930693
3,Robustness,Bootstrap CI for d_z,Primary uncertainty,"1.0783812409477092, 1.665422764129092"
4,Better than clinical scale?,d_z composite vs FARS/SARA,RQ1,1.3349658930315178 vs 0.728521346993977
5,Better than MRI alone?,vs strongest individual MRI feature,RQ1,1.3349658930315178 vs -0.9768059805067008
6,Disease specific?,FRDA vs control change,Specificity,NaN
7,Clinically meaningful?,Spearman Z vs FARS/SARA,RQ3,0.371772
8,Tracks clinical change?,Spearman delta Z vs delta FARS/SARA,Strong RQ3,-0.070271
9,Feature robustness,Coefficient/sign/Jaccard stability,Robustness,NaN


Saved one consolidated model-performance CSV: /Users/robertwang/Documents/New_project/biomarkers/results/model_performance_summary.csv


,model,question,metric,role,value,n,status,evidence,cv_mode,source
0,SRM Global Linear,2-year disease sensitivity,V1->V3 paired d_z,Primary,1.334966,101.0,computed,composite V1->V3,subject-level grouped 5-fold,srm_composite.ipynb
1,SRM Global Linear,Annual sensitivity,V1->V2 & V2->V3 d_z,Secondary,1.200079966720386; 0.4124137833891401,108.0,computed,composite V1->V2 and V2->V3,subject-level grouped 5-fold,srm_composite.ipynb
2,SRM Global Linear,Direction consistency,P(delta > 0),Secondary,0.930693,101.0,computed,composite V1->V3,subject-level grouped 5-fold,srm_composite.ipynb
3,SRM Global Linear,Robustness,bootstrap CI for d_z,Primary uncertainty,"1.0783812409477092, 1.665422764129092",101.0,computed,composite V1->V3 bootstrap CI,subject-level grouped 5-fold,srm_composite.ipynb
4,SRM Global Linear,Better than clinical scale?,d_z composite vs FARS/SARA,RQ1,1.3349658930315178 vs 0.728521346993977,101.0,computed,clinical interval benchmark,subject-level grouped 5-fold,srm_composite.ipynb
5,SRM Global Linear,Better than MRI alone?,vs strongest individual MRI feature,RQ1,1.3349658930315178 vs cerebellumFS: -0.9768059...,101.0,computed,strongest single MRI feature,subject-level grouped 5-fold,srm_composite.ipynb
6,SRM Global Linear,Disease specific?,FRDA vs control change,Specificity,NaN,NaN,missing,FRDA vs control change,subject-level grouped 5-fold,srm_composite.ipynb
7,SRM Global Linear,Clinically meaningful?,Spearman Z vs FARS/SARA,RQ3,0.371772,202.0,computed,cross-sectional Spearman,subject-level grouped 5-fold,srm_composite.ipynb
8,SRM Global Linear,Tracks clinical change?,Spearman delta Z vs delta FARS/SARA,Strong RQ3,-0.070271,101.0,computed,delta Spearman,subject-level grouped 5-fold,srm_composite.ipynb
9,SRM Global Linear,Feature robustness,coefficient/sign/Jaccard stability,Robustness,NaN,NaN,missing,stability diagnostics,subject-level grouped 5-fold,srm_composite.ipynb
